Step 1: Load and prepare documents

In [22]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = ["https://lilianweng.github.io/posts/2023-06-23-agent/"]

# Load documents from the URLs
docs = [WebBaseLoader(
                url, 
                requests_kwargs={"headers": {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}}
            ).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

Step 2: Split documents into chunks

In [23]:
# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)
# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

Step 3: Create a vector store

In [24]:
from langchain_community.vectorstores import SKLearnVectorStore  
from langchain_openai import AzureOpenAIEmbeddings

# Azure OpenAI Configuration  
azure_openai_endpoint = "https://oai-aifoundry-dev-eus.openai.azure.com/"
azure_openai_api_key = "Fw6FS1sOk6Rd96YwQQHq3TaiCS2XTiyqHpRrXqAonCDHdJif83NZJQQJ99CCACYeBjFXJ3w3AAABACOGiUNm"  # Get from Azure Portal
azure_openai_api_version = "2024-08-01-preview"

vectorstore = SKLearnVectorStore.from_documents(
            documents=doc_splits,
            embedding=AzureOpenAIEmbeddings(
                azure_endpoint=azure_openai_endpoint,
                azure_deployment="text-embedding-3-small",
                openai_api_key=azure_openai_api_key,
                openai_api_version=azure_openai_api_version,
            ),
        )
      
retriever = vectorstore.as_retriever(k=4)
print(f"\n✅ Vector store created successfully!")


✅ Vector store created successfully!


In [25]:
retriever

VectorStoreRetriever(tags=['SKLearnVectorStore', 'AzureOpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.sklearn.SKLearnVectorStore object at 0x000001E42B45AD20>, search_kwargs={})

Step 4: Set up the LLM and prompt template

In [27]:
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

# Define the prompt template for the LLM
prompt = PromptTemplate(
    template="""You are an assistant for question-answering tasks.
    Use the following documents to answer the question.
    If you don't know the answer, just say that you don't know.
    Use three sentences maximum and keep the answer concise:
    Question: {question}
    Documents: {documents}
    Answer:
    """,
    input_variables=["question", "documents"],
)

llm = AzureChatOpenAI(
    azure_endpoint=azure_openai_endpoint,
    azure_deployment="gpt-4o-mini",
    openai_api_key=azure_openai_api_key,
    openai_api_version=azure_openai_api_version,
    temperature=0,
)
# Create a chain combining the prompt template and LLM
rag_chain = prompt | llm | StrOutputParser()

# Equivalent explicit forms: 
# rag_chain = RunnableSequence(prompt, llm, StrOutputParser())
# rag_chain = prompt.chain(llm).chain(StrOutputParser())


Step 5: Integrate the retriever and LLM into a RAG application

In [28]:
# Define the RAG application class
class RAGApplication:
    def __init__(self, retriever, rag_chain):
        self.retriever = retriever
        self.rag_chain = rag_chain
        
    def run(self, question):
        # Retrieve relevant documents
        documents = self.retriever.invoke(question)
        # Extract content from retrieved documents
        doc_texts = "\\n".join([doc.page_content for doc in documents])
        # Get the answer from the language model
        answer = self.rag_chain.invoke({"question": question, "documents": doc_texts})
        return answer

# Check if both components were set up successfully
if 'vectorstore' in locals() and vectorstore is not None and 'rag_chain' in locals() and 'rag_chain' in locals():
    print("✅ All components ready! RAG Application can be created.")
else:
    print("🚨 Cannot create RAG application - some components failed to initialize.")
    if 'vectorstore' not in locals() or vectorstore is None:
        print("   - Vector store creation failed")
    if 'rag_chain' not in locals():
        print("   - RAG chain creation failed")
    print("\nPlease fix the Azure OpenAI configuration issues above before proceeding.")

✅ All components ready! RAG Application can be created.


Step 6: Test your RAG application

In [30]:
# Create and test the RAG application
if 'vectorstore' in locals() and vectorstore is not None and 'rag_chain' in locals():
    rag_app = RAGApplication(retriever, rag_chain)

    # Test with some questions
    questions = [
        "What are the key components of an AI agent?",
        "How do adversarial attacks work on large language models?"
    ]

    # Test each question
    for question in questions:
        print(f"\n🤖 Question: {question}")
        print("-" * 50)
        try:
            answer = rag_app.run(question)
            print(f"📋 Answer: {answer}")
        except Exception as e:
            print(f"❌ Error: {e}")
        print("=" * 50)
else:
    print("🚨 Cannot test RAG application - setup incomplete.")
    print("Please run the previous cells to fix configuration issues.")


🤖 Question: What are the key components of an AI agent?
--------------------------------------------------
📋 Answer: The key components of an AI agent include planning, memory, and tool use. Planning involves breaking down tasks into subgoals and self-reflection for improvement. Memory consists of short-term and long-term capabilities, while tool use allows the agent to access external information and APIs.

🤖 Question: How do adversarial attacks work on large language models?
--------------------------------------------------
📋 Answer: I don't know.
